# Interact with particles thanks to pose estimation

## Import modules

In [1]:
# Import internal modules
# import math
from pathlib import Path
# import random
from typing import Dict, List, Optional, Set, Tuple, TypedDict

# Import 3rd party modules
import cv2
import numpy as np

# Import local modules
from pose_estimation.play_with_particles.particle import Particle
from pose_estimation.play_with_particles.environment import Environment

from core.utils.renderer.resizer import resize_with_crop
from utils.project_manager import Project

## Set up project

In [2]:
# create project
project = Project(project_dir="assets/images/pose_estimation")

# Define constants

In [16]:
CAPTION: str = "play_with_particles"
NB_PARTICLES = 1000
PARTICLE_MIN_SIZE: int = 2
PARTICLE_MAX_SIZE: int = 8
PARTICLE_MASS = 50
THICKNESS = -1

In [22]:
# set input & output video path
video_path = Path("assets/images/gagu/gagu_teaser.mp4")
out_path = project.project_dir / f"{video_path.stem}_{CAPTION}.mp4"

# Initialize video stream
video_cap = cv2.VideoCapture(str(video_path))

# get video parameters
video_nb_frames = int(video_cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_fps = video_cap.get(cv2.CAP_PROP_FPS)
video_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"number of frames = {video_nb_frames}")
print(f"fps = {video_fps}")
print(f"video width = {video_width}")
print(f"video height = {video_height}")

# set codec for output video
codec = "H264"

rotate = False
resize = False

# set output shape
# out_height, out_width, out_channel = 1920, 1080, 3
out_height, out_width, out_channel = video_height, video_width, 3

# create a videoWriter object
fourcc = cv2.VideoWriter_fourcc(*codec)
out_video = cv2.VideoWriter(filename=str(out_path), fourcc=fourcc, fps=video_fps, frameSize=(out_width, out_height))

# Instantiate environment
world = Environment((video_width, video_height))
world.gravity = (np.pi, 0.1)

# read first video frame to get particles colors
_, first_frame = video_cap.read()

# Instantiate particles
particles_list: List[Particle] = []
for particle_nb in range(NB_PARTICLES):

    # set random position in frame
    x_px = np.random.randint(0, world.width_px)
    y_px = np.random.randint(0, world.height_px)

    # set random size
    size_px = np.random.randint(PARTICLE_MIN_SIZE, PARTICLE_MAX_SIZE)

    # set color to particles: mean of region of interest in frame (position where particles initially start)
    # get roi in frame
    roi = first_frame[y_px - size_px//2:y_px + size_px//2, x_px - size_px//2:x_px + size_px//2]
    # get mean of roi
    roi_mean = cv2.mean(roi)[:-1]

    particle = Particle(
        str(particle_nb),
        (x_px, y_px),
        size_px=size_px,
        mass=PARTICLE_MASS, # ToDo: play with color intensity to represent mass
        color=roi_mean,
        thickness=THICKNESS
        ) 
    particles_list.append(particle)

# while True:
for _ in range(25):

    # read video stream
    ret, frame = video_cap.read()

    # break out of loop if empty frame
    if not ret:
        print(f"frame is empty")
        break

    # move particles
    for particle in particles_list:
        particle.move()
        world.add_air_resistance(particle)
        # world.attraction(player_1, particle)
        # world.collide(particle, player_1, True)
        world.bounce(particle)
        for particle_2 in particles_list:
            if particle.name != particle_2.name:
                world.collide(particle, particle_2, True)

        particle.angle, particle.speed = world.add_vectors((particle.angle, particle.speed), world.gravity)
        
        # limit particle speed
        if particle.speed > 20:
                particle.speed = 20

    # draw particles
    for particle in particles_list:
        frame = cv2.circle(frame, (int(particle.x_px), int(particle.y_px)), particle.size_px, particle.color, particle.thickness)

    # rotate & resize frame if asked
    if rotate:
        frame = np.rot90(frame, -1)
    
    if resize:
        frame = resize_with_crop(frame, ref_img_shape=(out_height, out_width, out_channel))

    # write output frame
    out_video.write(frame)

# release video stream & video rendering
video_cap.release()
out_video.release()

number of frames = 1580
fps = 30.000765842334793
video width = 720
video height = 1280


In [ ]:
# ============================================================
# Run
# ============================================================

if __name__ == '__main__':

    

    # Instantiate player
    player_1 = Player(
        "John Titor",
        (100, world.height_px/2),
        size_px=PLAYER_SIZE,
        mass=100,
        color=becode_color
    )

            
            world.attraction(player_1, aibot)
        
                collide_obstacle_aibot: bool = world.collide(obstacle, aibot, False)
                # If collision, punish the aibot and remove it
                if collide_player or collide_obstacle_aibot:
                    
        # Move obstacles
        for obstacle in obstacles_list:
            obstacle.move()
            world.add_air_resistance(obstacle)
            world.collide(obstacle, player_1, True)
            world.bounce(obstacle)

            # Limits obstacle's speed
            if obstacle.speed > 20:
                obstacle.speed = 20